# 157 — Diseño de evaluaciones y criterios de éxito

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** La semilla forma parte de **T** (el protocolo): fija la parte estocástica del
procedimiento para que la medición sea reproducible. No es dato (D), ni métrica (M), ni criterio (C).


In [ ]:
result = run_lab("evaluation", seed=157)
assert result["kind"] == "evaluation"
assert result["evidence"]
show(result)


**Ejercicio 2.** Con 500 correos, 25 phishing, filtro marca 40 y acierta 20:

```text
accuracy_baseline = 475/500 = 0.95   (predecir siempre "legítimo")
TP=20, FP=20, FN=5, TN=455
accuracy_filtro   = (20+455)/500 = 0.95  (¡igual al baseline!)
recall            = 20/25 = 0.80
precision         = 20/40 = 0.50
```

La métrica principal debe ser el **recall** de phishing: el costo dominante es dejar pasar un
ataque. La accuracy es idéntica al baseline trivial y por tanto no discrimina.

**Ejercicio 3.** Hipótesis: contaminación — el modelo memorizó los ítems literales del benchmark
(presentes en su corpus) y la paráfrasis rompe la coincidencia superficial sin cambiar el
constructo. Confirmación: probar con ítems nuevos escritos después de la fecha de corte del
entrenamiento, o perturbaciones mínimas (renombrar variables, reordenar opciones); si el
rendimiento se mantiene bajo, la memorización explica el 95 % original.

**Ejercicio 4.** Ejemplo: un resumidor es malo en fidelidad factual pero produce textos fluidos.
Tras ver resultados, el equipo reporta ROUGE (alto, premia solapamiento léxico) y omite la tasa de
afirmaciones no soportadas (alta). El sistema "aprueba" una métrica que no mide el constructo
crítico (fidelidad). Por eso M y C se fijan antes de medir.


In [ ]:
# Verificación numérica del Ejercicio 2
total, phishing = 500, 25
TP, FP = 20, 20
FN, TN = phishing - TP, total - phishing - FP
accuracy_baseline = (total - phishing) / total
accuracy_filtro = (TP + TN) / total
recall = TP / phishing
precision = TP / (TP + FP)
print(f"baseline={accuracy_baseline:.3f} filtro={accuracy_filtro:.3f} "
      f"recall={recall:.2f} precision={precision:.2f}")
assert abs(accuracy_baseline - accuracy_filtro) < 1e-9
assert recall == 0.80 and precision == 0.50


## Reflexión (guía)

1. Chequeo de contaminación: correr los mismos ítems parafraseados o ítems posteriores a la fecha
   de corte; una caída grande indica memorización.
2. Declarar C antes de medir evita el *metric shopping*: si el criterio se elige después, siempre
   existe alguna métrica favorable y la evaluación pierde poder de refutación.
3. Respuesta abierta: lo esencial es identificar el error más caro (FN vs FP) y elegir métrica
   principal + guardarraíl coherentes con ese costo.
